# Comparison between FastSDP and Conic Bundle results

In [1]:
import numpy as np
import pandas as pd

In [2]:
folder = "/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-1/2026_08_11_17h41_45s_test-comparison-conic-bundle-untargeted/part_0_100"
fastsdp_file = f"{folder}/results.csv"
conic_bundle_file = f"{folder}/results_conic_bundle.csv"

In [3]:
fastsdp_csv = pd.read_csv(fastsdp_file)
fastsdp_csv = fastsdp_csv.rename(columns={'target': 'target', 'data_index': 'data_index', 
                                          'time': 'time_fastsdp', 'status': 'status_fastsdp', 
                                        'objective': 'objective_fastsdp', 'optimal_value': 'optimal_value_fastsdp'})
fastsdp_csv = fastsdp_csv[fastsdp_csv['status_fastsdp'].isin(['optimal', 'infeasible', 'unbounded', 1, 0, '1', '0'])]
fastsdp_csv['fastsdp_time'] = pd.to_numeric(fastsdp_csv['time_fastsdp'], errors='coerce')
fastsdp_csv['optimal_value_fastsdp'] = pd.to_numeric(fastsdp_csv['optimal_value_fastsdp'], errors='coerce')

conic_bundle_csv = pd.read_csv(conic_bundle_file)
conic_bundle_csv = conic_bundle_csv.rename(columns={'target': 'target', 'data_index': 'data_index', 
                                                    'time': 'time_conic_bundle', 'status': 'status_conic_bundle', 
                                                    'objective': 'objective_conic_bundle', 'optimal_value': 'optimal_value_conic_bundle'})
conic_bundle_csv = conic_bundle_csv[conic_bundle_csv['status_conic_bundle'].isin(['optimal', 'infeasible', 'unbounded', 1, 0, '1', '0'])]
conic_bundle_csv['conic_bundle_time'] = pd.to_numeric(conic_bundle_csv['time_conic_bundle'], errors='coerce')
conic_bundle_csv['optimal_value_conic_bundle'] = pd.to_numeric(conic_bundle_csv['optimal_value_conic_bundle'], errors='coerce')

data_indexed_fastsdp = fastsdp_csv["data_index"].unique()
data_indexed_conic_bundle = conic_bundle_csv["data_index"].unique()
data_indexed = np.intersect1d(data_indexed_fastsdp, data_indexed_conic_bundle)

conic_bundle_csv = conic_bundle_csv[conic_bundle_csv['data_index'].isin(data_indexed)]
fastsdp_csv = fastsdp_csv[fastsdp_csv['data_index'].isin(data_indexed)]

In [4]:
is_all_nan_target = conic_bundle_csv['target'].isna().all() and fastsdp_csv['target'].isna().all()
if is_all_nan_target:
    print("All target values are NaN.")
elif conic_bundle_csv['target'].isna().any() or fastsdp_csv['target'].isna().any():
    print("Warning: Some target values are NaN. This may affect the comparison.")
else : 
    print("All target values are valid.")

All target values are NaN.


In [5]:
csv_all = pd.merge(fastsdp_csv, conic_bundle_csv, on=['data_index', 'target'], how='inner', suffixes=('_fastsdp', '_conic_bundle'))

In [6]:
csv_all.columns

Index(['network_fastsdp', 'model_fastsdp', 'dataset_fastsdp', 'data_index',
       'label_fastsdp', 'label_predicted', 'target', 'epsilon_fastsdp',
       'status_fastsdp', 'CHORDAL_DECOMPOSITION', 'LAST_LAYER',
       'USE_STABLE_ACTIVES', 'USE_STABLE_INACTIVES', 'Nb_stable_inactives',
       'Nb_stable_actives', 'Nb_constraints', 'Nb_variables', 'RLT',
       'triangularization', 'McCormick_beta_z', 'beta_logits_comparaison',
       'beta_logits_comparaison_big_M', 'sum_beta_logits_equal_logit',
       'RLT_prop', 'iterations', 'time_fastsdp', 'pretreatment_time',
       'bound_time', 'optimal_value_fastsdp', 'primal_obj_value',
       'dual_obj_value', 'fastsdp_time', 'network_conic_bundle',
       'model_conic_bundle', 'dataset_conic_bundle', 'label_conic_bundle',
       'epsilon_conic_bundle', 'status_conic_bundle',
       'optimal_value_conic_bundle', 'is_robust', 'time_conic_bundle',
       'n_iter_cb', 'dynamic', 'dualize', 'conic_bundle_time'],
      dtype='str')

In [7]:
csv_all['optimal_value_diff'] = csv_all['optimal_value_fastsdp'] - csv_all['optimal_value_conic_bundle']
csv_all['time_diff'] = csv_all['fastsdp_time'] - csv_all['conic_bundle_time']

In [10]:
csv_all[[ 'optimal_value_diff', 'n_iter_cb', 'time_diff']].max()

optimal_value_diff      3.197421
n_iter_cb             500.000000
time_diff              -0.602845
dtype: float64

In [12]:
csv_all[csv_all['optimal_value_diff'].abs() > 1e-2][['data_index', 'target', 'optimal_value_fastsdp', 'optimal_value_conic_bundle', 'n_iter_cb', 'time_diff']].shape

(73, 6)